In [ ]:

# ===== PXR FOUNDATION: neural Hill-equation dose-response model (GPU) =====
import numpy as np, pandas as pd, glob, time, os, traceback, torch, torch.nn as nn

def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if not hits:
        for r, _, fs in os.walk('/kaggle/input'):
            for ff in fs: print('  ', os.path.join(r, ff))
        raise FileNotFoundError(name)
    return sorted(hits, key=len)[0]

try:
    comp = pd.read_parquet(find('found_compounds.parquet')).sort_values('comp_id').reset_index(drop=True)
    feat = pd.read_parquet(find('found_features.parquet')).sort_values('comp_id').reset_index(drop=True)
    crc  = pd.read_parquet(find('found_crc.parquet'))
    cnt  = pd.read_parquet(find('found_counter.parquet'))
    sc   = pd.read_parquet(find('found_sc.parquet'))
    def pick_device():
        if torch.cuda.is_available():
            cc = torch.cuda.get_device_capability(0)
            if cc[0] >= 7:
                return 'cuda'
            print(f'GPU cc {cc} < 7.0 (no kernel image) -> CPU fallback')
        return 'cpu'
    dev  = pick_device()
    print('device', dev, '| compounds', len(comp), '| crc', len(crc), '| counter', len(cnt), '| sc obs', len(sc))

    # test_pos -> comp_id, derived from found_compounds (no json needed)
    n_test = int(comp['test_pos'].max()) + 1
    tc = np.full(n_test, -1, dtype=int)
    tm = comp[comp['test_pos'] >= 0]
    tc[tm['test_pos'].astype(int).values] = tm['comp_id'].astype(int).values
    print('test positions', n_test, '| covered', int((tc >= 0).sum()))

    X = feat.drop(columns='comp_id').values.astype('float32')
    mu, sd = X.mean(0), X.std(0) + 1e-6
    Xt = torch.tensor((X - mu) / sd, device=dev)
    N, Dh = Xt.shape

    def col(df, c, dt=torch.float32):
        return torch.tensor(df[c].values, dtype=dt, device=dev)
    crc_id = col(crc, 'comp_id', torch.long); crc_y = col(crc, 'pec50')
    crc_em = col(crc, 'emax'); crc_em_mask = torch.isfinite(crc_em)
    cnt_id = col(cnt, 'comp_id', torch.long); cnt_y = col(cnt, 'pec50_null')
    sc_id  = col(sc, 'comp_id', torch.long); sc_pc = col(sc, 'pconc'); sc_fc = col(sc, 'log2fc'); sc_w = col(sc, 'w')
    LN10 = float(np.log(10.0))

    class Hill(nn.Module):
        def __init__(self, d, h=512, p=0.2):
            super().__init__()
            self.enc = nn.Sequential(
                nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(p),
                nn.Linear(h, h // 2), nn.BatchNorm1d(h // 2), nn.ReLU(), nn.Dropout(p))
            self.pxr  = nn.Linear(h // 2, 3)
            self.null = nn.Linear(h // 2, 1)
        def forward(self, x):
            z = self.enc(x); o = self.pxr(z)
            return (o[:, 0], nn.functional.softplus(o[:, 1]),
                    nn.functional.softplus(o[:, 2]) + 0.2, self.null(z).squeeze(-1))

    A_EMAX, B_NULL, G_SC, EPOCHS, LR, WD, ENS = 0.3, 0.3, 1.0, 300, 1e-3, 1e-4, 6

    def train_one(seed):
        torch.manual_seed(seed); np.random.seed(seed)
        m = Hill(Dh).to(dev)
        opt = torch.optim.Adam(m.parameters(), lr=LR, weight_decay=WD)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
        for ep in range(EPOCHS):
            m.train(); opt.zero_grad()
            pec50, emax, n, null = m(Xt)
            L_crc  = ((pec50[crc_id] - crc_y) ** 2).mean()
            L_emax = ((emax[crc_id][crc_em_mask] - crc_em[crc_em_mask]) ** 2).mean()
            L_null = ((null[cnt_id] - cnt_y) ** 2).mean()
            pe, em, nn_ = pec50[sc_id], emax[sc_id], n[sc_id]
            R = em * torch.sigmoid(LN10 * nn_ * (pe - sc_pc))
            L_sc = (sc_w * (R - sc_fc) ** 2).mean()
            loss = L_crc + A_EMAX * L_emax + B_NULL * L_null + G_SC * L_sc
            loss.backward(); opt.step(); sch.step()
            if ep % 100 == 0:
                print(f'  seed{seed} ep{ep} crc{L_crc.item():.3f} emax{L_emax.item():.3f} null{L_null.item():.3f} sc{L_sc.item():.3f}')
        m.eval()
        with torch.no_grad():
            pec50, emax, n, null = m(Xt)
        return (pec50.cpu().numpy(), emax.cpu().numpy(), n.cpu().numpy(), null.cpu().numpy())

    t0 = time.time()
    preds = [train_one(s) for s in range(ENS)]
    pec50 = np.mean([p[0] for p in preds], 0); emax = np.mean([p[1] for p in preds], 0)
    nhill = np.mean([p[2] for p in preds], 0); null = np.mean([p[3] for p in preds], 0)
    print(f'trained {ENS} seeds in {time.time()-t0:.0f}s')

    def gather(arr): return np.array([arr[c] if c >= 0 else np.nan for c in tc], dtype='float32')
    out = pd.DataFrame({'test_pos': range(len(tc)), 'pec50': gather(pec50),
                        'emax': gather(emax), 'nhill': gather(nhill), 'null': gather(null)})
    out.to_parquet('/kaggle/working/found_pred_513.parquet', index=False)
    np.save('/kaggle/working/found_pec50_513.npy', gather(pec50))
    print('saved found_pred_513.parquet | covered', int(out.pec50.notna().sum()), '/', len(tc))
    print(out.describe())
except Exception:
    open('/kaggle/working/ERROR.txt', 'w').write(traceback.format_exc())
    print(traceback.format_exc()); raise
